# Human-Swarm Interaction with SHASTA

**Notebook 1 of 2.** We simulate a UAV/UGV swarm on a real city map, control it with a **config file**, command it the way
a human operator would, and record the interaction (the operator's orders **and eye gaze**) with Lab Streaming Layer (LSL) into an XDF file.

| Step | Topic |
|---|---|
| **1** | Check the environment |
| **2** | Load a map: the street graph |
| **3** | The config file, actors and the 3D scene |
| **4** | Order one group to a node |
| **5** | Two groups: formations and observations |
| **6** | How the config changes the result |
| **7** | Human in the loop: `SwarmCommander` and the GUI |
| **8** | Write our own experiment |
| **9** | LSL basics with `pylsl` |
| **10** | Record order markers and eye gaze to XDF |
| **11** | Load the XDF with `pyxdf` and analyse |

Select the **Python (smc)** kernel (top right). Keep `config.yaml` and `lsl_tools.py` in the same folder as this notebook.

## Step 1: Check the environment

JupyterHub runs on a server with no screen. Two settings let the graphical libraries run anyway:

- **PyBullet** (the physics engine) runs in `DIRECT` mode: no window, but it can still render images.
- **pygame** (the GUI) draws on SDL's `dummy` video driver: an in-memory surface instead of a window.

The environment variables must be set **before** pygame is imported.

In [ ]:
import os, sys
os.environ["SDL_VIDEODRIVER"] = "dummy"   # pygame draws off-screen
os.environ["SDL_AUDIODRIVER"] = "dummy"   # no sound card on the server
sys.path.insert(0, os.getcwd())           # so we can import lsl_tools.py from this folder

Now check that every library we need is importable, and print its version.

In [ ]:
import importlib

libs = ["numpy", "matplotlib", "pygame", "pybullet", "shasta", "pylsl", "pyxdf", "yaml"]
for name in libs:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", None) or getattr(getattr(module, "version", None), "ver", "")
        print(f"ok       {name:11s} {version}")
    except Exception as exc:
        print(f"MISSING  {name:11s} {type(exc).__name__}: {exc}")

Every line should say `ok`. If `pylsl` fails with a message about `liblsl`, the LSL system library is missing.
Ask the hub admin to run `conda install -c conda-forge liblsl` in the shared environment.

Finally, check that PyBullet really runs without a display, and see which maps SHASTA can find.

In [ ]:
import pybullet as p
client = p.connect(p.DIRECT)          # DIRECT = physics only, no window
print("PyBullet connected, client id:", client)
p.disconnect()

from shasta.assets import list_maps
print("Maps available:", list_maps())

**Checkpoint.** All libraries show `ok`, PyBullet connected, and at least `buffalo-small` is listed.

## Step 2: Load a map

SHASTA builds its world from **OpenStreetMap** data. A map folder holds the street network and the buildings,
plus a small table that converts latitude/longitude to the simulator's metre coordinates.

The `Map` class can be used on its own, with no physics needed. We give it the name of a map.

In [ ]:
from shasta.map import Map

world_map = Map()
world_map.setup({"map_to_use": "buffalo-small"})
print("Map loaded")

### The street graph

Roads are stored as a **graph**: *nodes* are intersections and *edges* are the streets between them.
Everything we do later, planning routes and ordering a swarm to go somewhere, is expressed in these nodes.

In [ ]:
graph = world_map.get_node_graph()
print("intersections (nodes):", graph.number_of_nodes())
print("street segments (edges):", graph.number_of_edges())

Every node has a position. Let's look at one node in the simulator's metre coordinates, and in latitude/longitude.

In [ ]:
node = 5 if 5 in graph else next(iter(graph.nodes))
print("node:", node)
print("simulator (x, y, z) metres:", world_map.get_cartesian_node_position(node).round(1))
print("latitude, longitude:       ", world_map.get_lat_lon_node_position(node))

### Draw the map: buildings, streets and numbered intersections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

def draw_map(ax, world_map, label_nodes=True):
    """Buildings (grey), streets (blue) and, optionally, numbered intersections."""
    for outline in world_map.get_building_footprints():
        ax.add_patch(Polygon(outline, closed=True, facecolor="#c9ccd3", edgecolor="#8d93a1", linewidth=0.6))
    for segment in world_map.get_street_segments():
        ax.plot(segment[:, 0], segment[:, 1], color="#3b6fb6", linewidth=1.2)
    if label_nodes:
        for n in world_map.get_node_graph().nodes:
            x, y = world_map.get_cartesian_node_position(n)[:2]
            ax.plot(x, y, "o", color="white", markeredgecolor="black", markersize=9)
            ax.text(x, y, str(n), ha="center", va="center", fontsize=6)
    ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")

fig, ax = plt.subplots(figsize=(9, 9))
draw_map(ax, world_map)
ax.set_title("buffalo-small: buildings (grey), streets (blue), intersections (numbered)")
plt.show()

**Checkpoint.** You should see a small city block with numbered intersections. In the next steps we will order swarms
from one node to another.

*Try it:* change `"buffalo-small"` to another name from `list_maps()` in Step 1 and run the cells again.
(Only `buffalo-small` ships inside the package. Other maps need the `SHASTA_ASSETS` folder.)

## Step 3: The config file, actors and the 3D scene

A simulation is described by a **config file**: a short YAML text file next to this notebook. The notebook reads
`config.yaml`, builds everything from it, and later steps change values in it to show what each setting does.
Here it is:

In [ ]:
CONFIG_PATH = "config.yaml"
print(open(CONFIG_PATH).read())

SHASTA's `load_config` reads a YAML file into a dictionary (and lets you override top-level keys):

In [ ]:
from shasta.config import load_config

config = load_config(CONFIG_PATH)
for section, value in config.items():
    print(f"{section:12s} {value}")

Which part is read by whom:

| Keys | Read by | Meaning |
|---|---|---|
| `headless`, `time_step`, `experiment.map_to_use` | SHASTA | window or not, physics step, which map |
| `swarm` | this notebook | how many vehicles, where they start, how fast they are |
| `mission` | this notebook | targets, reach radius, time limit |
| `human`, `collector` | this notebook | the recorded session, the experiment of Step 8 |

### Actors, groups and the environment

An **actor** is one vehicle: a `UAV` (drone, flies) or a `UGV` (ground vehicle, drives). Actors are organised in
**groups**: a dictionary `{group_id: [actors]}`. A group is what a human commands.

The **environment** (`ShastaEnv`) combines the map, the PyBullet physics world and an *experiment*, which defines what the actors
do and what we observe. We use `GoToNodeExperiment`: "go to a map node, in formation".

In [ ]:
from shasta.actors import UAV, UGV
from shasta.env import ShastaEnv
from shasta.experiments import GoToNodeExperiment

def vehicle_group(kind, n, node, speed):
    """``n`` vehicles of type ``kind`` in a small ring around map node ``node``, each limited to ``speed`` m/s."""
    lat, lon = world_map.get_lat_lon_node_position(node)
    vehicles = []
    for angle in (np.linspace(0, 2 * np.pi, n, endpoint=False) if n > 1 else [0.0]):
        ring = 5e-5 if n > 1 else 0.0                                   # about 5 m, in degrees
        vehicle = kind(init_pos=[lat + ring * np.sin(angle), lon + ring * np.cos(angle)])
        vehicle.speed = speed
        vehicles.append(vehicle)
    return vehicles

def build_groups(config, uavs=None, ugvs=None):
    """The actor groups described by ``config['swarm']``: group 0 = UAVs, group 1 = UGVs."""
    s = config["swarm"]
    return {0: vehicle_group(UAV, uavs or s["uavs"], s["uav_start_node"], s["uav_speed"]),
            1: vehicle_group(UGV, ugvs or s["ugvs"], s["ugv_start_node"], s["ugv_speed"])}

def make_env(groups, config=None, experiment=GoToNodeExperiment, experiment_config=None):
    """Build a ShastaEnv from a config dictionary (default: the one read from config.yaml)."""
    import copy
    config = copy.deepcopy(config if config is not None else load_config(CONFIG_PATH))
    config["experiment"] = {**config.get("experiment", {}), "type": experiment}
    if experiment_config is not None:
        config["experiment"]["config"] = experiment_config
    return ShastaEnv(config, groups)

env = make_env(build_groups(config, uavs=1, ugvs=1))
obs, info = env.reset()
for group_id, positions in obs.items():
    print(f"group {group_id}: positions (x, y, z) in metres ->", positions.round(1).tolist())

The **observation** is a dictionary `{group_id: array of shape (n_actors, 3)}`: the position of every vehicle. Because the config fixes the start
nodes, the vehicles start in the same place on every run.

Each actor also has properties such as its type, speed and battery.

In [ ]:
for group_id, actors in env.core.get_actor_groups().items():
    a = actors[0]
    print(f"group {group_id}: {a.type}  speed={a.speed} m/s  battery={a.battery:.0f}  ammo={a.ammo}")

### Look at the 3D scene

PyBullet can render what a camera sees, even with no window. `getCameraImage` returns an image with the
**TinyRenderer**, a software renderer that needs no GPU. We wrap it in a small helper that looks at a point from above.

In [ ]:
def camera_frame(env, center_xy, size=(640, 480), distance=90.0, yaw=0.0, pitch=-55.0):
    """Render the PyBullet scene as a (height, width, 3) image, looking at ``center_xy`` (metres)."""
    client = env.core.physics_client
    projection = client.computeProjectionMatrixFOV(60, size[0] / size[1], 0.5, 2000)
    view = client.computeViewMatrixFromYawPitchRoll(
        [float(center_xy[0]), float(center_xy[1]), 5.0], distance, yaw, pitch, 0, 2)
    w, h, rgba, *_ = client.getCameraImage(
        *size, viewMatrix=view, projectionMatrix=projection, renderer=client.ER_TINY_RENDERER)
    return np.reshape(np.asarray(rgba, dtype=np.uint8), (h, w, 4))[:, :, :3]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (group_id, title) in zip(axes, [(0, "UAV group"), (1, "UGV group")]):
    ax.imshow(camera_frame(env, obs[group_id].mean(axis=0)[:2])); ax.set_title(title); ax.axis("off")
plt.show()

And the same two vehicles on the map from Step 2, seen from above:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
draw_map(ax, world_map, label_nodes=False)
for group_id, marker, color, label in [(0, "^", "tab:blue", "UAV"), (1, "s", "tab:red", "UGV")]:
    ax.plot(*obs[group_id][0, :2], marker, color=color, markersize=12, label=label)
ax.legend(); ax.set_title("Where the vehicles start (set by config.yaml)")
plt.show()

**Checkpoint.** Two vehicles in the 3D view and two markers on the map. Next we make one of them move.

## Step 4: Order one group to a node

`env.step(actions)` advances the simulation by one tick. The action is a dictionary `{group_id: node}`.
After the first order we call `env.step(None)`, which means "keep going". The environment plans a route along the streets
and the group flies or drives it. The episode is `done` when the group's centre is within 3 m of its target.

We send the UAV to the node **farthest** from where it starts, so the trip is long enough to see.

In [ ]:
def farthest_node(world_map, xy):
    nodes = list(world_map.get_node_graph().nodes)
    return max(nodes, key=lambda n: np.linalg.norm(world_map.get_cartesian_node_position(n)[:2] - xy))

env.close()
env = make_env(build_groups(config, uavs=1, ugvs=1))
obs, _ = env.reset()
start = obs[0].mean(axis=0)[:2]
target = farthest_node(world_map, start)
print("start (x, y):", start.round(1), " -> target node", target, world_map.get_cartesian_node_position(target)[:2].round(1))

trace = [start]
obs, reward, done, truncated, info = env.step({0: target})    # the order
steps = 1
while not done and steps < 20000:
    obs, reward, done, truncated, info = env.step(None)       # keep going
    trace.append(obs[0].mean(axis=0)[:2]); steps += 1
trace = np.array(trace)
print(f"done={done} after {steps} steps; travelled {np.linalg.norm(np.diff(trace, axis=0), axis=1).sum():.0f} m")

The recorded path, drawn on the map:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
draw_map(ax, world_map, label_nodes=False)
ax.plot(trace[:, 0], trace[:, 1], color="tab:orange", linewidth=2.5, label="UAV path")
ax.plot(*trace[0], "o", color="green", markersize=12, label="start")
ax.plot(*world_map.get_cartesian_node_position(target)[:2], "*", color="red", markersize=18, label=f"target node {target}")
ax.legend(); ax.set_title("One group, one order")
plt.show()

A camera view at the start, halfway and at the end of the trip. The camera looks at the UAV, so we replay the same trip and capture three moments.

In [ ]:
env.close()
env = make_env(build_groups(config, uavs=1, ugvs=1))
obs, _ = env.reset()
capture = {1: "start", len(trace) // 2: "halfway", len(trace) - 2: "arrived"}
views = {}
obs, *_ = env.step({0: target})
for i in range(1, len(trace)):
    if i in capture:
        views[capture[i]] = camera_frame(env, obs[0].mean(axis=0)[:2], distance=60)
    obs, *_ = env.step(None)

fig, axes = plt.subplots(1, len(views), figsize=(5 * len(views), 4))
for ax, (label, img) in zip(np.atleast_1d(axes), views.items()):
    ax.imshow(img); ax.set_title(label); ax.axis("off")
plt.show()

**Checkpoint.** A route that follows the streets from the start to the red star, and three camera views of the drone along the way.

## Step 5: Two groups, formations and observations

Now the full swarm from the config: a **UAV group** (`swarm.uavs` drones) and a **UGV group** (`swarm.ugvs` ground vehicles).
We order both at once: `env.step({0: node_a, 1: node_b})`. The vehicles of a group keep a **formation** around the group's centre
while they follow the planned route.

In [ ]:
env.close()
env = make_env(build_groups(config))
obs, _ = env.reset()
print("observation shapes:", {g: v.shape for g, v in obs.items()})

rng = np.random.default_rng(1)
nodes = list(world_map.get_node_graph().nodes)
targets = {0: int(rng.choice(nodes)), 1: int(rng.choice(nodes))}
print("targets:", targets)

history = {g: [obs[g].copy()] for g in obs}
obs, reward, done, _, _ = env.step(targets)
steps = 1
while not done and steps < 30000:
    obs, reward, done, _, _ = env.step(None); steps += 1
    if steps % 10 == 0:
        for g in obs:
            history[g].append(obs[g].copy())
print("done:", done, "after", steps, "steps")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
draw_map(ax, world_map, label_nodes=False)
colors = {0: "tab:blue", 1: "tab:red"}
for g, snapshots in history.items():
    arr = np.array(snapshots)                      # (time, n_actors, 3)
    for k in range(arr.shape[1]):
        ax.plot(arr[:, k, 0], arr[:, k, 1], color=colors[g], alpha=0.6, linewidth=1.2)
    ax.plot(*world_map.get_cartesian_node_position(targets[g])[:2], "*", color=colors[g], markersize=18,
            markeredgecolor="black", label=f"group {g} target")
ax.legend(); ax.set_title("The whole swarm, two orders")
plt.show()

for g in history:
    arr = np.array(history[g])
    spread = np.linalg.norm(arr[:, :, :2] - arr[:, :, :2].mean(axis=1, keepdims=True), axis=2).mean(axis=1)
    print(f"group {g}: mean distance to the group's centre: start {spread[0]:.1f} m, middle {spread[len(spread)//2]:.1f} m")

**Checkpoint.** Paths in two colours that end at the two stars. The *spread* numbers show the formation:
the vehicles keep a roughly constant distance from their group's centre.

Everything so far was set by `config.yaml`. In the next step we change the config and measure what changes.

## Step 6: How the config changes the result

We keep **everything the same** (the map, the start node, the target node) and change **one config value at a time**, then
measure the same trip. Because the start is fixed by the config, any difference comes from the setting we changed.

First, a helper that edits a value in a copy of the config and runs one trip of a single UAV to a fixed target:

In [ ]:
import copy
import pandas as pd

def edit(config, section, key, value):
    """A copy of ``config`` with ``config[section][key] = value`` (or ``config[section] = value`` if key is None)."""
    new = copy.deepcopy(config)
    if key is None:
        new[section] = value
    else:
        new[section][key] = value
    return new

TRIP_TARGET = 57 if 57 in world_map.get_node_graph() else target

def run_trip(cfg, uavs=1, ugvs=0):
    """Order group 0 (UAVs, or the UGV if uavs=0) to TRIP_TARGET. Returns steps, battery left and formation spread."""
    kind_group = build_groups(cfg, uavs=max(uavs, 1), ugvs=max(ugvs, 1))
    groups = {0: kind_group[0] if uavs else kind_group[1]}
    env = make_env(groups, config=cfg)
    obs, _ = env.reset()
    obs, _, done, *_ = env.step({0: TRIP_TARGET}); steps = 1
    while not done and steps < 40000:
        obs, _, done, *_ = env.step(None); steps += 1
    spread = float(np.linalg.norm(obs[0][:, :2] - obs[0][:, :2].mean(axis=0), axis=1).mean())
    battery = float(np.mean([a.battery for a in env.core.get_actor_groups()[0]]))
    env.close()
    return {"steps": steps, "battery_left": round(battery, 1), "formation_spread_m": round(spread, 1), "arrived": done}

baseline = run_trip(config)
print("baseline (config.yaml as it is):", baseline)

### 1. `swarm.uav_speed`: how fast the vehicles go

Speed is the most a vehicle can move per step. We try four values for the same trip:

In [ ]:
speeds = [1.0, 2.5, 5.0, 10.0]
rows = [{"uav_speed": v, **run_trip(edit(config, "swarm", "uav_speed", v))} for v in speeds]
speed_table = pd.DataFrame(rows)
display(speed_table)

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(speed_table["uav_speed"], speed_table["steps"], "o-")
ax.set_xlabel("config swarm.uav_speed (m/s)"); ax.set_ylabel("steps to arrive"); ax.set_title("Twice the speed, half the steps")
plt.tight_layout(); plt.show()

### 2. The vehicle type: UAV or UGV

Same trip, same speed, different vehicle. UAVs use more battery per metre than UGVs, so after the same trip they have less left:

In [ ]:
vehicle_rows = [{"vehicle": "UAV", **run_trip(config, uavs=1, ugvs=0)}, {"vehicle": "UGV", **run_trip(config, uavs=0, ugvs=1)}]
display(pd.DataFrame(vehicle_rows)[["vehicle", "steps", "battery_left"]])

### 3. `time_step`: a setting that does *not* matter here

`time_step` is a real SHASTA setting (it sets PyBullet's physics step), so it is natural to expect it to change the trip. Let's check instead of assuming:

In [ ]:
rows = [{"time_step": v, **run_trip(edit(config, "time_step", None, v))} for v in (0.5, 1.0, 2.0, 4.0)]
display(pd.DataFrame(rows)[["time_step", "steps", "battery_left"]])

The number of steps is identical for every value. In this simulator a vehicle's motion each step is set by its speed, not by the physics step, so
`time_step` has no effect on the trip. **Not every key in a config file changes every outcome.** Test a setting before you rely on it.

### 4. Group size: `swarm.uavs`

More drones in the group changes the **formation**, not the trip time:

In [ ]:
rows = [{"uavs": n, **run_trip(edit(config, "swarm", "uavs", n), uavs=n)} for n in (1, 4, 8)]
display(pd.DataFrame(rows)[["uavs", "steps", "formation_spread_m"]])

### 5. `mission.time_limit` and `mission.radius`: the human's task

These two keys change the **mission** the human has to complete: how long they have and how close a group has to get to a target.
We use the scripted human (it always gives the next target to a free group) and read the final score.

In [ ]:
from shasta.interaction import Mission, SwarmCommander

def scripted_human(commander, pending):
    """Give every free group the next unassigned target."""
    for group_id in commander.groups:
        free = group_id not in commander.orders or commander.status(group_id)["arrived"]
        if pending and free:
            commander.send(group_id, pending.pop(0))

def make_mission(world_map, cfg):
    m = cfg["mission"]
    return Mission.random(world_map, n_targets=m["targets"], seed=m["seed"], radius=m["radius"], time_limit=m["time_limit"])

def run_mission(cfg, max_steps=20000):
    env = make_env(build_groups(cfg), config=cfg)
    env.reset()
    mission = make_mission(env.core.get_map(), cfg)
    commander = SwarmCommander(env, mission, steps_per_update=1)
    pending = list(mission.targets)
    while not mission.is_over(commander.step_count) and commander.step_count < max_steps:
        scripted_human(commander, pending)
        commander.update()
    env.close()
    return {"score": f"{mission.score}/{len(mission.targets)}", "steps_used": commander.step_count}

rows = [{"time_limit": v, **run_mission(edit(config, "mission", "time_limit", v))} for v in (300, 800, 1500, 2500)]
print("Changing mission.time_limit (steps):")
display(pd.DataFrame(rows))

In [ ]:
rows = [{"radius_m": v, **run_mission(edit(config, "mission", "radius", v))} for v in (3.0, 12.0, 40.0)]
print("Changing mission.radius (metres):")
display(pd.DataFrame(rows))

**Checkpoint.** You have seen three kinds of effect from one text file:

- **Large:** `uav_speed` changes how long a trip takes, and `time_limit` decides whether the human completes the mission at all.
- **Moderate:** the vehicle type changes the battery used, and `radius` shifts the finishing step a little.
- **None:** `time_step` changed nothing.

*Try it:* edit `config.yaml` (for example `uav_speed: 5.0` or `time_limit: 800`), then use **Kernel > Restart & Run All**
and see the later steps change too.

## Step 7: Human in the loop

`SwarmCommander` is the layer a human (or a GUI, or an AI advisor) talks to. It can

- **select** groups and **order** them to nodes (`commander.send(group, node)`),
- **advance** the simulation (`commander.update()`),
- report **status** (position, battery, distance to target, ETA) and an **event log**.

A **Mission** is a list of target nodes, built from the `mission` section of the config. The mission is complete when every target has been reached by some group.

In [ ]:
env.close()
env = make_env(build_groups(config))
env.reset()
mission = make_mission(env.core.get_map(), config)
commander = SwarmCommander(env, mission, steps_per_update=config["human"]["steps_per_update"])
print("mission:", mission)

commander.send(0, mission.targets[0])            # the human orders group 0 to the first target
for _ in range(100):
    commander.update()
print(commander.status(0))
print(commander.events)

### The GUI, drawn off-screen

`shasta gui` is a pygame window built on top of exactly this `SwarmCommander`. Here we create the same `ShastaGUI`
on pygame's dummy driver and read its drawing surface. A helper turns the surface into an image.

In [ ]:
import pygame
from shasta.gui import ShastaGUI

def gui_frame(gui):
    """The GUI's current window as a (height, width, 3) image."""
    return np.asarray(pygame.surfarray.array3d(gui.screen)).swapaxes(0, 1)

gui = ShastaGUI(commander)
for _ in range(3):
    gui.step()
fig = plt.figure(figsize=(13, 8)); plt.imshow(gui_frame(gui)); plt.axis("off"); plt.title("The human interface, off-screen")
plt.show()

### A scripted "human"

A real operator would click. For the tutorial the `scripted_human` function from Step 6 plays the operator: whenever a group is free it sends it to
the next target that has not been assigned. We run the loop through the GUI so we can take snapshots.

In [ ]:
env.close()
env = make_env(build_groups(config))
env.reset()
mission = make_mission(env.core.get_map(), config)
commander = SwarmCommander(env, mission, steps_per_update=config["human"]["steps_per_update"])
gui = ShastaGUI(commander)
pending = list(mission.targets)

snapshots, frame = {}, 0
while not mission.is_over(commander.step_count) and frame < 3000:
    scripted_human(commander, pending)
    gui.step()
    frame += 1
    if frame in (5, 80, 160):
        snapshots[frame] = gui_frame(gui).copy()
snapshots["end"] = gui_frame(gui).copy()
print(f"{frame} frames. Mission score: {mission.score}/{len(mission.targets)}")
for step, text in commander.events:
    print(f"  step {step:5d}: {text}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (key, img) in zip(axes.ravel(), snapshots.items()):
    ax.imshow(img); ax.set_title(f"frame {key}"); ax.axis("off")
plt.tight_layout(); plt.show()

**Checkpoint.** The snapshots show the swarm groups moving, the target markers being reached, and the event log filling.
This is exactly what a participant sees in a human-swarm study. In Steps 10 and 11 we record the participant's **orders and eye gaze** while they do it.

## Step 8: Write our own experiment

So far we used the ready-made `GoToNodeExperiment`. A research study needs its own rules: what the actors do,
what we observe, **when the episode ends** and **how performance is scored**. An experiment is a class that fills in four methods
of `BaseExperiment`:

| Method | Question it answers |
|---|---|
| `apply_actions(actions, core)` | What do the vehicles do this tick? |
| `get_observation(observation, core)` | What does the agent or human get to see? |
| `get_done_status(observation, core)` | Is the episode over? |
| `compute_reward(observation, core)` | How well did it go this tick? |

`GoToNodeExperiment` already does the first two (route planning and flying in formation), so we **extend it** and add our own
done condition and reward. Our experiment, **TargetCollector**: *one UAV group must visit a list of nodes, one after another.
It gets +1 for every node reached and −0.001 for every step it takes.*

How it picks the **next node** is a *strategy* we pass in through the experiment config. That makes the experiment a fair way to
compare strategies. The number of nodes, the number of episodes and which strategies to compare come from the `collector` section of `config.yaml`.

In [ ]:
class TargetCollector(GoToNodeExperiment):
    """One group (id 0) visits ``exp_config['nodes']``; the next node is chosen by ``exp_config['strategy']``."""

    def reset(self):
        super().reset()
        self.targets, self.paths = {}, {}
        self.remaining = list(self.exp_config["nodes"])
        self.visited, self.current, self.arrived_now = [], None, False

    def _centroid(self, core):
        actors = core.get_actors_by_group_id(0)
        return np.mean([a.get_pos_and_orientation()[0] for a in actors], axis=0)[:2]

    def _reached(self, centroid, core):
        node_xy = core.get_map().get_cartesian_node_position(self.current)[:2]
        return np.linalg.norm(centroid - node_xy) <= self.tolerance

    def apply_actions(self, actions, core):
        self.arrived_now = False
        centroid = self._centroid(core)
        if self.current is not None and self._reached(centroid, core):
            self.visited.append(self.current); self.current, self.arrived_now = None, True
        if self.current is None and self.remaining:
            self.current = self.exp_config["strategy"](centroid, self.remaining, core.get_map())
            self.remaining.remove(self.current)
            self.set_target(0, self.current)            # plan the route (inherited)
            if len(self.paths[0]) == 0:                 # already at the start of the route: go straight there
                self.paths[0] = np.array([core.get_map().get_cartesian_node_position(self.current)])
        super().apply_actions(None, core)               # fly the route in formation (inherited)

    def get_done_status(self, observation, core):
        if self.current is not None and not self.remaining and self._reached(self._centroid(core), core):
            self.visited.append(self.current); self.current, self.arrived_now = None, True
        return self.current is None and not self.remaining and len(self.visited) > 0

    def compute_reward(self, observation, core):
        return (1.0 if self.arrived_now else 0.0) - 0.001

### Two strategies to compare

A strategy is a function `(centroid, remaining_nodes, world_map) -> next_node`.

- **nearest**: always fly to the closest remaining node (straight-line distance).
- **random**: visit them in a random order.

In [ ]:
def nearest(centroid, remaining, world_map):
    return min(remaining, key=lambda n: np.linalg.norm(world_map.get_cartesian_node_position(n)[:2] - centroid))

def make_random(seed):
    rng = np.random.default_rng(seed)
    return lambda centroid, remaining, world_map: remaining[int(rng.integers(len(remaining)))]

STRATEGIES = {"nearest": lambda seed: nearest, "random": make_random}

def run_episode(nodes, strategy, cfg=config, max_steps=60000):
    s = cfg["swarm"]
    group = vehicle_group(UAV, 3, s["uav_start_node"], s["uav_speed"])
    for vehicle in group:
        vehicle.battery = 1000        # the default 100 lasts about 2000 m; visiting five nodes can need more
    env = make_env({0: group}, config=cfg, experiment=TargetCollector, experiment_config={"nodes": nodes, "strategy": strategy})
    obs, _ = env.reset()
    trace, total_reward, done, steps = [obs[0].mean(axis=0)[:2]], 0.0, False, 0
    while not done and steps < max_steps:
        obs, reward, done, _, _ = env.step(None)
        total_reward += reward; steps += 1
        trace.append(obs[0].mean(axis=0)[:2])
    visited = list(env.experiment.visited)
    env.close()
    trace = np.array(trace)
    return {"reward": total_reward, "steps": steps, "visited": visited,
            "distance_m": float(np.linalg.norm(np.diff(trace, axis=0), axis=1).sum()), "trace": trace, "done": done}

col = config["collector"]
nodes_demo = [int(n) for n in np.random.default_rng(3).choice(list(world_map.get_node_graph().nodes), col["n_nodes"], replace=False)]
print("nodes to visit:", nodes_demo)
result_nearest = run_episode(nodes_demo, nearest)
result_random = run_episode(nodes_demo, make_random(0))
for name, r in (("nearest", result_nearest), ("random", result_random)):
    print(f"{name:8s} done={r['done']}  visit order={r['visited']}  steps={r['steps']}  distance={r['distance_m']:.0f} m  reward={r['reward']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, r) in zip(axes, (("nearest", result_nearest), ("random", result_random))):
    draw_map(ax, world_map, label_nodes=False)
    ax.plot(r["trace"][:, 0], r["trace"][:, 1], color="tab:orange", linewidth=2)
    for order, n in enumerate(r["visited"], start=1):
        x, y = world_map.get_cartesian_node_position(n)[:2]
        ax.plot(x, y, "o", color="red", markersize=14); ax.text(x, y, str(order), color="white", ha="center", va="center", fontsize=8)
    ax.set_title(f"{name}: {r['distance_m']:.0f} m, reward {r['reward']:.2f}")
plt.show()

One node set is an anecdote. To compare strategies fairly we run several random node sets and look at the average.
The number of episodes comes from `collector.episodes` in the config.

In [ ]:
rows = []
all_nodes = list(world_map.get_node_graph().nodes)
for seed in range(col["episodes"]):
    nodes = [int(n) for n in np.random.default_rng(seed).choice(all_nodes, col["n_nodes"], replace=False)]
    for name in col["strategies"]:
        r = run_episode(nodes, STRATEGIES[name](seed))
        rows.append({"seed": seed, "strategy": name, "steps": r["steps"], "distance_m": r["distance_m"], "reward": r["reward"], "finished": r["done"]})
table = pd.DataFrame(rows)
print("episodes that did not finish:", int((~table["finished"]).sum()), "of", len(table))
print(table.groupby("strategy")[["steps", "distance_m", "reward"]].agg(["mean", "std"]).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
table.pivot(index="seed", columns="strategy", values="distance_m").plot.bar(ax=ax)
ax.set_ylabel("distance flown (m)"); ax.set_title("Same nodes, two strategies")
plt.tight_layout(); plt.show()

**Checkpoint.** `nearest` should fly a shorter distance and score a higher reward than `random` for most node sets, and every episode should finish.
That is a complete experiment: our own rules, a reward, a done condition and a measured comparison.

*Try it:* change `collector.episodes` or `n_nodes` in `config.yaml` and re-run, or add a third strategy to `STRATEGIES`.

The part we have left out is the **human**. Next: how to *record* a human working with the swarm.

## Step 9: LSL basics with `pylsl`

**Lab Streaming Layer (LSL)** is the standard way to move time-stamped data between programs and devices: eye trackers, EEG, task software.
Everyone publishes a **stream** and anyone on the network can read it, with all streams sharing **one clock**, so they can be lined up later.

- An **outlet** publishes samples. A stream has a *name*, a *type* (`Markers`, `Gaze`, `EEG`, ...), a number of *channels*,
  a *sampling rate* (`IRREGULAR_RATE` for events) and a *data format*.
- An **inlet** finds a stream and reads samples, each with an LSL **timestamp**.

Below, one program is both sides, which is the simplest way to see how it works.

In [ ]:
import time, pylsl

info = pylsl.StreamInfo(name="demo-events", type="Markers", channel_count=1,
                        nominal_srate=pylsl.IRREGULAR_RATE, channel_format="string", source_id="demo-1")
outlet = pylsl.StreamOutlet(info)

found = pylsl.resolve_byprop("name", "demo-events", timeout=5)
print("streams found:", [(s.name(), s.type(), s.channel_count()) for s in found])

inlet = pylsl.StreamInlet(found[0])
inlet.open_stream()

t0 = pylsl.local_clock()
for word in ["start", "order group 0", "order group 1", "end"]:
    outlet.push_sample([word])
    time.sleep(0.25)

samples, stamps = inlet.pull_chunk(timeout=1.0)
for sample, stamp in zip(samples, stamps):
    print(f"t = {stamp - t0:5.2f} s   {sample[0]}")

Every sample carries a timestamp from LSL's own clock (`pylsl.local_clock()`). That clock is what lets us line up
**orders (from SHASTA)** and **gaze (from the eye tracker)**, even though they come from different programs.

A regular-rate numeric stream, like a 60 Hz eye tracker, works the same way:

In [ ]:
info2 = pylsl.StreamInfo("demo-signal", "Demo", 1, 50, "float32", "demo-2")
outlet2 = pylsl.StreamOutlet(info2)
inlet2 = pylsl.StreamInlet(pylsl.resolve_byprop("name", "demo-signal", timeout=5)[0]); inlet2.open_stream()

t_start = time.perf_counter()
n = 0
while time.perf_counter() - t_start < 2.0:
    outlet2.push_sample([float(np.sin(2 * np.pi * 1.0 * (time.perf_counter() - t_start)))]); n += 1
    time.sleep(1 / 50)
values, stamps = inlet2.pull_chunk(timeout=1.0)
values, stamps = np.array(values)[:, 0], np.array(stamps)
print(f"received {len(stamps)} samples, about {len(stamps) / (stamps[-1] - stamps[0]):.1f} Hz")
plt.figure(figsize=(7, 2.5)); plt.plot(stamps - stamps[0], values, "."); plt.xlabel("time (s)"); plt.title("A 50 Hz stream, read through LSL")
plt.show()
del outlet, outlet2, inlet, inlet2

**Checkpoint.** The events arrive with timestamps about 0.25 s apart, and the numeric stream is read back at close to 50 Hz.

## Step 10: Record order markers and eye gaze to XDF

Now we record a human-swarm session. We need three things:

1. A **marker stream** from SHASTA: every order the human gives, with where the target is *on the screen*.
2. A **gaze stream** from an eye tracker: where the human is looking, in screen pixels.
3. A **recorder** that saves both into one **XDF** file (the standard LSL recording format).

### Which gaze source?

| Source | Where it runs | When to use it |
|---|---|---|
| **Real Tobii** (`tobii_to_lsl.py`) | on the **laptop** the tracker is plugged into | the real study. LabRecorder records into XDF; upload the file and go to Step 11 |
| **Synthetic** (`SyntheticGaze`) | here, on the hub | a live demo with no hardware. The stream is marked `synthetic = true` |

A USB eye tracker cannot be plugged into a remote server, and LSL finds streams by network broadcast on the local network,
which usually does not cross to the hub. So for the real study the whole recording runs on the laptop
(`shasta_gui_lsl.py` + `tobii_to_lsl.py` + LabRecorder), and **analysis** happens here. The cell below runs the same pipeline
on the hub using synthetic gaze, so we can walk through everything without hardware.

In [ ]:
from lsl_tools import MarkerOutlet, SyntheticGaze, Recorder, attach_markers, attach_order_markers, screen_xy

SESSION_SECONDS = config["human"]["session_seconds"]       # from config.yaml
XDF_PATH = "session.xdf"

env.close()
env = make_env(build_groups(config))
env.reset()
mission = make_mission(env.core.get_map(), config)
commander = SwarmCommander(env, mission, steps_per_update=config["human"]["steps_per_update"])
gui = ShastaGUI(commander)
view = gui.view
pending = list(mission.targets)

# 1. Event markers from SHASTA, with each order's target position on the screen
markers = MarkerOutlet("SHASTA-Events")
attach_markers(commander, markers)
attach_order_markers(commander, markers, view)

# 2. Gaze. Synthetic here; with the real tracker this is a stream from tobii_to_lsl.py instead.
gaze = SyntheticGaze(width=gui.size[0], height=gui.size[1]).start()

# 3. The recorder: finds every stream of type Markers and Gaze and writes one XDF file
recorder = Recorder(XDF_PATH, stream_types=("Markers", "Gaze")).start()
print("recording:", recorder.streams)

Now the session. The scripted human plays the operator. Between orders, the synthetic gaze **looks at the target** the operator just ordered,
then **follows the group** on the map. It runs at 30 frames per second so that gaze and markers line up in real time (the length comes from `human.session_seconds`).

In [ ]:
markers.push("session_start", screen=list(gui.size), map_rect=list(gui.map_rect), targets=mission.targets,
             synthetic_gaze=True)

look_hold, last_orders, frame, start = 0, 0, 0, time.time()
session_frame = None
while time.time() - start < SESSION_SECONDS and not mission.is_over(commander.step_count):
    scripted_human(commander, pending)

    if len(commander.events) != last_orders:                        # a new order or arrival happened
        last_orders = len(commander.events)
        if commander.orders:
            newest = list(commander.orders.values())[-1]
            gaze.look_at(*screen_xy(view, commander.node_position(newest)))   # look at the target
            look_hold = 25                                                  # for ~0.8 s
    if look_hold > 0:
        look_hold -= 1
    else:
        group = commander.selected
        gaze.look_at(*screen_xy(view, commander.status(group)["centroid"]))   # then follow the group

    gui.step()
    gui.clock.tick(30)                                              # real-time pacing, 30 frames per second
    frame += 1
    if frame == 120:
        session_frame = gui_frame(gui).copy()

markers.push("session_end", score=mission.score, frames=frame)
saved = recorder.stop()
gaze.stop()
env.close()
if session_frame is not None:
    plt.imsave("session_frame.png", session_frame)
print(f"{frame} frames, mission score {mission.score}/{len(mission.targets)}")
print("saved", saved, f"({os.path.getsize(saved) / 1024:.0f} KB)")

### Recording the real thing (on the laptop)

For the actual study, everything runs on the laptop that has the eye tracker. Files are in this folder:

```text
1. python tobii_to_lsl.py --width 1920 --height 1080      # Tobii -> LSL "Gaze" stream
2. python shasta_gui_lsl.py --map buffalo-small           # the real GUI + LSL "Markers" stream
3. LabRecorder: Update, select both streams, Start, then run the task, then Stop.
```

LabRecorder writes the `.xdf` file. Upload it next to this notebook and set `XDF_PATH` in the next step.
The files in `tobii_to_lsl.py` and `shasta_gui_lsl.py` have not been run against real hardware in this repo, so check them
on your own setup before the tutorial.

## Step 11: Load the XDF with `pyxdf` and analyse

`pyxdf.load_xdf` reads every stream in the file together with its timestamps. Our helper `load_streams` returns them by type.
Set `XDF_PATH` to your own LabRecorder file to analyse a real session. Nothing else changes.

In [ ]:
from lsl_tools import load_streams, gaze_table, marker_table, detect_fixations, gaze_heatmap

XDF_PATH = "session.xdf"          # <- change to your LabRecorder file for a real recording
streams = load_streams(XDF_PATH)

for stream in streams.values():
    span = stream["ts"][-1] - stream["ts"][0]
    print(f"{stream['type']:8s} {stream['name']:16s} {len(stream['ts']):6d} samples over {span:5.1f} s"
          f" ({len(stream['ts']) / span:5.1f} Hz){'   <-- SYNTHETIC data' if stream['synthetic'] else ''}")

Both streams share one clock, so we can put them on a common time axis, starting at zero. We split the markers by event type.

In [ ]:
gaze_data = gaze_table(streams["Gaze"])
events = marker_table(streams["Markers"])

t0 = min(gaze_data["t"][0], events[0]["t"])
gt = gaze_data["t"] - t0
for e in events: e["t"] -= t0

start = next(e for e in events if e["event"] == "session_start")
SCREEN = tuple(start["screen"])                     # (width, height) in pixels
MAP_RECT = start["map_rect"]                        # x, y, w, h of the map panel
orders = [e for e in events if e["event"] == "order"]

print("screen:", SCREEN, "  orders:", len(orders), "  valid gaze samples:", f"{gaze_data['valid'].mean():.0%}")
pd.DataFrame(orders)[["t", "group", "node", "screen_x", "screen_y"]].round(1)

### Gaze over time, with the operator's orders

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(gt, gaze_data["x"], label="gaze x (px)", linewidth=0.8)
ax.plot(gt, gaze_data["y"], label="gaze y (px)", linewidth=0.8)
for e in orders:
    ax.axvline(e["t"], color="red", linestyle="--", alpha=0.7)
    ax.text(e["t"], ax.get_ylim()[1], f" order g{e['group']}", color="red", va="top", fontsize=8)
ax.set_xlabel("time (s)"); ax.set_ylabel("screen position (px)"); ax.legend(loc="lower right")
plt.show()

### Where did the operator look?

A **heatmap** of all valid gaze samples on top of a screenshot of the GUI, and the **fixations** found with a dispersion-threshold algorithm
(gaze that stays within 40 px for at least 100 ms), drawn as circles whose size shows how long each fixation lasted.

In [ ]:
fixations = detect_fixations(gaze_data["t"], gaze_data["x"], gaze_data["y"], max_dispersion=40, min_duration=0.10)
print(len(fixations), "fixations, median duration", f"{np.median([f['end'] - f['start'] for f in fixations]) * 1000:.0f} ms")

heat = gaze_heatmap(gaze_data["x"], gaze_data["y"], shape=(SCREEN[1], SCREEN[0]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
background = plt.imread("session_frame.png") if os.path.exists("session_frame.png") else None
for ax in axes:
    if background is not None:
        ax.imshow(background, extent=[0, SCREEN[0], SCREEN[1], 0])
    ax.set_xlim(0, SCREEN[0]); ax.set_ylim(SCREEN[1], 0); ax.axis("off")
axes[0].imshow(heat, extent=[0, SCREEN[0], SCREEN[1], 0], cmap="hot", alpha=0.55); axes[0].set_title("Gaze heatmap")
for f in fixations:
    axes[1].scatter(f["x"], f["y"], s=(f["end"] - f["start"]) * 3000, facecolors="none", edgecolors="cyan", linewidths=1.5)
axes[1].set_title("Fixations (circle size = duration)")
plt.tight_layout(); plt.show()

### Reaction to each order

For every order, how long did it take until a fixation landed **on the target** that had just been ordered
(within 60 px of its screen position)? And how long did the eyes stay on it?

In [ ]:
rows = []
for e in orders:
    target_px = np.array([e["screen_x"], e["screen_y"]])
    later = [f for f in fixations if f["start"] - t0 >= e["t"]]
    hit = next((f for f in later if np.linalg.norm([f["x"], f["y"]] - target_px) <= 60), None)
    rows.append({"order_t": e["t"], "group": e["group"], "node": e["node"],
                 "latency_s": (hit["start"] - t0 - e["t"]) if hit else np.nan,
                 "dwell_s": (hit["end"] - hit["start"]) if hit else np.nan})
reaction = pd.DataFrame(rows).round(2)
display(reaction)
print("mean latency to look at the target:", reaction["latency_s"].mean().round(2), "s")

on_map = ((gaze_data["x"] >= MAP_RECT[0]) & (gaze_data["x"] <= MAP_RECT[0] + MAP_RECT[2]) &
          (gaze_data["y"] >= MAP_RECT[1]) & (gaze_data["y"] <= MAP_RECT[1] + MAP_RECT[3]))
print(f"share of valid gaze on the map panel: {on_map[gaze_data['valid']].mean():.0%}")

## Summary

| What we did | Tool |
|---|---|
| Loaded a real city as a street graph | `shasta.map.Map` |
| Described the simulation in a YAML file and measured what each setting does | `config.yaml`, `load_config` |
| Simulated UAV/UGV groups and rendered the 3D scene headless | `ShastaEnv`, PyBullet `DIRECT`, `getCameraImage` |
| Let a human command the swarm | `SwarmCommander`, `ShastaGUI` (off-screen) |
| Defined our own experiment and compared strategies | `BaseExperiment` subclass |
| Streamed orders and gaze with LSL and saved them as XDF | `pylsl`, `lsl_tools.Recorder` |
| Loaded the recording and measured gaze against orders | `pyxdf`, fixations, reaction time |

**Next:** *Notebook 2* does the same recording and analysis with MOSAIC, the grid-based human-AI search-and-rescue task.